# 14 — Patient-Level Prediction

This notebook converts respiratory-cycle predictions into patient-level
disease predictions.

The disease diagnosis is associated with the patient, while individual
respiratory cycles provide the model inputs. Therefore, treating every
cycle
as an independent disease diagnosis can overrepresent patients with many
recorded cycles.

Two aggregation strategies are evaluated:

1. Majority voting across cycle predictions.
2. Mean probability aggregation across cycle predictions.

The evaluation uses the patient-disjoint CNN test split established in
Notebook 11.

The Lightweight Mel CNN is used because it achieved the strongest
balanced accuracy and macro F1 among the deep-learning experiments.

In [1]:
import json
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported.")

Libraries imported.


In [2]:
PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

DL_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "deep_learning"
)

CNN_BASELINE_DIR = (
    DL_DIR /
    "cnn_baseline"
)

PATIENT_LEVEL_DIR = (
    DL_DIR /
    "patient_level_prediction"
)

PATIENT_LEVEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(PATIENT_LEVEL_DIR)

Output directory:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction


In [3]:
class_mapping_path = (
    DL_DIR /
    "disease_class_mapping.csv"
)

class_mapping_df = pd.read_csv(
    class_mapping_path
)

class_names = (
    class_mapping_df
    .sort_values("disease_label")["diagnosis"]
    .tolist()
)

num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Classes:")

for i, name in enumerate(class_names):
    print(i, "->", name)

Number of classes: 8
Classes:
0 -> Asthma
1 -> Bronchiectasis
2 -> Bronchiolitis
3 -> COPD
4 -> Healthy
5 -> LRTI
6 -> Pneumonia
7 -> URTI


In [4]:
test_manifest_path = (
    CNN_BASELINE_DIR /
    "test_cnn_manifest.csv"
)

test_manifest_patient = pd.read_csv(
    test_manifest_path
)

print("Test cycles:", len(test_manifest_patient))
print(
    "Test patients:",
    test_manifest_patient["patient_id"].nunique()
)

print()
print(
    test_manifest_patient[
        [
            "patient_id",
            "recording_id",
            "diagnosis",
            "disease_label"
        ]
    ].head()
)

Test cycles: 1561
Test patients: 25

   patient_id recording_id diagnosis  disease_label
0         101          1b1      URTI              7
1         101          1b1      URTI              7
2         101          1b1      URTI              7
3         101          1b1      URTI              7
4         101          1b1      URTI              7


In [5]:
patient_label_counts = (
    test_manifest_patient
    .groupby("patient_id")["disease_label"]
    .nunique()
)

print(
    "Patients with multiple disease labels:",
    (patient_label_counts > 1).sum()
)

if (patient_label_counts > 1).sum() == 0:
    print("PASS: Every patient has exactly one disease label.")
else:
    print("WARNING: Some patients have multiple disease labels.")

Patients with multiple disease labels: 0
PASS: Every patient has exactly one disease label.


In [6]:
patient_truth_df = (
    test_manifest_patient[
        [
            "patient_id",
            "diagnosis",
            "disease_label"
        ]
    ]
    .drop_duplicates("patient_id")
    .sort_values("patient_id")
    .reset_index(drop=True)
)

print(
    "Patient-level ground truth shape:",
    patient_truth_df.shape
)

print(
    patient_truth_df["diagnosis"].value_counts()
)

Patient-level ground truth shape: (25, 3)
diagnosis
COPD              13
Healthy            5
URTI               3
Bronchiectasis     2
Pneumonia          1
Bronchiolitis      1
Name: count, dtype: int64


In [7]:
cnn_predictions_path = (
    CNN_BASELINE_DIR /
    "cnn_test_predictions.csv"
)

cnn_predictions_df = pd.read_csv(
    cnn_predictions_path
)

print(
    "Prediction rows:",
    len(cnn_predictions_df)
)

print(
    "Prediction columns:"
)

print(
    cnn_predictions_df.columns.tolist()
)

Prediction rows: 1561
Prediction columns:
['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'true_label', 'predicted_label', 'true_disease', 'predicted_disease']


In [8]:
print(
    cnn_predictions_df.head()
)

   patient_id recording_id chest_location diagnosis  disease_label  \
0         101          1b1             Al      URTI              7   
1         101          1b1             Al      URTI              7   
2         101          1b1             Al      URTI              7   
3         101          1b1             Al      URTI              7   
4         101          1b1             Al      URTI              7   

  official_split                                         audio_path cnn_split  \
0           test  /Users/vs/Sem 5/Respiratory-Disease-Classifica...      test   
1           test  /Users/vs/Sem 5/Respiratory-Disease-Classifica...      test   
2           test  /Users/vs/Sem 5/Respiratory-Disease-Classifica...      test   
3           test  /Users/vs/Sem 5/Respiratory-Disease-Classifica...      test   
4           test  /Users/vs/Sem 5/Respiratory-Disease-Classifica...      test   

   true_label  predicted_label true_disease predicted_disease  
0           7               

In [9]:
print(cnn_predictions_df.columns.tolist())

['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'true_label', 'predicted_label', 'true_disease', 'predicted_disease']


In [10]:
print("Cycle-level predictions:")
print(
    cnn_predictions_df[
        [
            "patient_id",
            "true_label",
            "true_disease",
            "predicted_label",
            "predicted_disease"
        ]
    ].head(10)
)

print()
print(
    "Cycle-level patient count:",
    cnn_predictions_df["patient_id"].nunique()
)

Cycle-level predictions:
   patient_id  true_label true_disease  predicted_label predicted_disease
0         101           7         URTI                5              LRTI
1         101           7         URTI                2     Bronchiolitis
2         101           7         URTI                2     Bronchiolitis
3         101           7         URTI                2     Bronchiolitis
4         101           7         URTI                2     Bronchiolitis
5         101           7         URTI                4           Healthy
6         101           7         URTI                4           Healthy
7         101           7         URTI                2     Bronchiolitis
8         101           7         URTI                2     Bronchiolitis
9         101           7         URTI                2     Bronchiolitis

Cycle-level patient count: 25


In [11]:
def majority_vote(group):
    counts = group["predicted_label"].value_counts()
    return counts.index[0]


patient_majority = (
    cnn_predictions_df
    .groupby("patient_id")
    .apply(
        majority_vote,
        include_groups=False
    )
    .reset_index(name="predicted_label")
)

print(
    "Patient-level majority predictions:",
    len(patient_majority)
)

print(
    patient_majority.head()
)

Patient-level majority predictions: 25
   patient_id  predicted_label
0         101                2
1         106                3
2         107                3
3         119                4
4         121                7


In [12]:
patient_majority_df = patient_majority.merge(
    patient_truth_df[
        [
            "patient_id",
            "disease_label",
            "diagnosis"
        ]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

patient_majority_df = patient_majority_df.rename(
    columns={
        "disease_label": "true_label",
        "diagnosis": "true_disease"
    }
)

patient_majority_df["predicted_disease"] = (
    patient_majority_df["predicted_label"]
    .map(
        dict(
            enumerate(class_names)
        )
    )
)

print(
    patient_majority_df.head(10)
)

   patient_id  predicted_label  true_label true_disease predicted_disease
0         101                2           7         URTI     Bronchiolitis
1         106                3           3         COPD              COPD
2         107                3           3         COPD              COPD
3         119                4           7         URTI           Healthy
4         121                7           4      Healthy              URTI
5         122                3           6    Pneumonia              COPD
6         125                7           4      Healthy              URTI
7         145                3           3         COPD              COPD
8         147                3           3         COPD              COPD
9         150                2           7         URTI     Bronchiolitis


In [13]:
print(
    "Patient rows:",
    len(patient_majority_df)
)

print(
    "Unique patients:",
    patient_majority_df["patient_id"].nunique()
)

print(
    "Missing true labels:",
    patient_majority_df["true_label"].isna().sum()
)

print(
    "Missing predictions:",
    patient_majority_df["predicted_label"].isna().sum()
)

Patient rows: 25
Unique patients: 25
Missing true labels: 0
Missing predictions: 0


In [14]:
y_true_patient = (
    patient_majority_df["true_label"]
    .astype(int)
    .to_numpy()
)

y_pred_patient_majority = (
    patient_majority_df["predicted_label"]
    .astype(int)
    .to_numpy()
)

patient_majority_accuracy = accuracy_score(
    y_true_patient,
    y_pred_patient_majority
)

patient_majority_balanced_accuracy = balanced_accuracy_score(
    y_true_patient,
    y_pred_patient_majority
)

patient_majority_macro_precision = precision_score(
    y_true_patient,
    y_pred_patient_majority,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

patient_majority_macro_recall = recall_score(
    y_true_patient,
    y_pred_patient_majority,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

patient_majority_macro_f1 = f1_score(
    y_true_patient,
    y_pred_patient_majority,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

patient_majority_weighted_f1 = f1_score(
    y_true_patient,
    y_pred_patient_majority,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:          {patient_majority_accuracy:.6f}")
print(f"Balanced Accuracy: {patient_majority_balanced_accuracy:.6f}")
print(f"Macro Precision:   {patient_majority_macro_precision:.6f}")
print(f"Macro Recall:      {patient_majority_macro_recall:.6f}")
print(f"Macro F1:          {patient_majority_macro_f1:.6f}")
print(f"Weighted F1:       {patient_majority_weighted_f1:.6f}")

Accuracy:          0.560000
Balanced Accuracy: 0.333333
Macro Precision:   0.132812
Macro Recall:      0.250000
Macro F1:          0.162069
Weighted F1:       0.482207


In [15]:
print(
    classification_report(
        y_true_patient,
        y_pred_patient_majority,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0
    )
)

                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00         2
 Bronchiolitis       0.25      1.00      0.40         1
          COPD       0.81      1.00      0.90        13
       Healthy       0.00      0.00      0.00         5
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.00      0.00      0.00         1
          URTI       0.00      0.00      0.00         3

      accuracy                           0.56        25
     macro avg       0.13      0.25      0.16        25
  weighted avg       0.43      0.56      0.48        25



In [16]:
patient_majority_cm = confusion_matrix(
    y_true_patient,
    y_pred_patient_majority,
    labels=np.arange(num_classes)
)

patient_majority_cm_df = pd.DataFrame(
    patient_majority_cm,
    index=class_names,
    columns=class_names
)

print(patient_majority_cm_df)

                Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  \
Asthma               0               0              0     0        0     0   
Bronchiectasis       0               0              0     2        0     0   
Bronchiolitis        0               0              1     0        0     0   
COPD                 0               0              0    13        0     0   
Healthy              0               0              1     0        0     0   
LRTI                 0               0              0     0        0     0   
Pneumonia            0               0              0     1        0     0   
URTI                 0               0              2     0        1     0   

                Pneumonia  URTI  
Asthma                  0     0  
Bronchiectasis          0     0  
Bronchiolitis           0     0  
COPD                    0     0  
Healthy                 0     4  
LRTI                    0     0  
Pneumonia               0     0  
URTI                    0

In [17]:
import torch
import torch.nn as nn

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


class LightweightMelCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

Device: mps


In [20]:
class LightweightMelCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [22]:
checkpoint = torch.load(
    best_cnn_path,
    map_location="cpu"
)

state_dict = checkpoint["model_state_dict"]

print("Checkpoint classifier parameters:")
for key, value in state_dict.items():
    if key.startswith("classifier"):
        print(
            f"{key}: shape={tuple(value.shape)}"
        )

Checkpoint classifier parameters:
classifier.3.weight: shape=(8, 64)
classifier.3.bias: shape=(8,)


In [23]:
print("All checkpoint keys:")
for key, value in state_dict.items():
    print(f"{key}: {tuple(value.shape)}")

All checkpoint keys:
features.0.weight: (16, 1, 3, 3)
features.0.bias: (16,)
features.1.weight: (16,)
features.1.bias: (16,)
features.1.running_mean: (16,)
features.1.running_var: (16,)
features.1.num_batches_tracked: ()
features.4.weight: (32, 16, 3, 3)
features.4.bias: (32,)
features.5.weight: (32,)
features.5.bias: (32,)
features.5.running_mean: (32,)
features.5.running_var: (32,)
features.5.num_batches_tracked: ()
features.8.weight: (64, 32, 3, 3)
features.8.bias: (64,)
features.9.weight: (64,)
features.9.bias: (64,)
features.9.running_mean: (64,)
features.9.running_var: (64,)
features.9.num_batches_tracked: ()
classifier.3.weight: (8, 64)
classifier.3.bias: (8,)


In [24]:
class LightweightMelCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [25]:
model_patient = LightweightMelCNN(
    num_classes=num_classes
).to(device)

best_cnn_path = (
    CNN_BASELINE_DIR /
    "best_lightweight_mel_cnn.pt"
)

checkpoint = torch.load(
    best_cnn_path,
    map_location=device
)

model_patient.load_state_dict(
    checkpoint["model_state_dict"]
)

model_patient.eval()

print("Best Lightweight Mel CNN loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Validation accuracy:", checkpoint["val_accuracy"])

Best Lightweight Mel CNN loaded successfully.
Best epoch: 32
Validation loss: 1.2070167528983018
Validation accuracy: 0.6159737417943107


In [26]:
normalization_path = (
    DL_DIR /
    "logmel_train_normalization.csv"
)

normalization_df = pd.read_csv(
    normalization_path
)

logmel_mean = float(
    normalization_df.loc[
        normalization_df["statistic"] == "mean",
        "value"
    ].iloc[0]
)

logmel_std = float(
    normalization_df.loc[
        normalization_df["statistic"] == "std",
        "value"
    ].iloc[0]
)

print("Log-Mel mean:", logmel_mean)
print("Log-Mel std :", logmel_std)

Log-Mel mean: -66.74081420898438
Log-Mel std : 17.33708953857422


In [27]:
import soundfile as sf
import librosa

TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = 20000

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
FMIN = 0
FMAX = 2000


def fix_audio_length(signal, target_samples):
    if len(signal) < target_samples:
        signal = np.pad(
            signal,
            (0, target_samples - len(signal)),
            mode="constant"
        )
    else:
        signal = signal[:target_samples]

    return signal.astype(np.float32)


def load_processed_audio(audio_path):
    signal, sr = sf.read(audio_path)

    if signal.ndim > 1:
        signal = np.mean(signal, axis=1)

    if sr != TARGET_SR:
        raise ValueError(
            f"Expected {TARGET_SR} Hz, got {sr} Hz"
        )

    return fix_audio_length(
        signal.astype(np.float32),
        TARGET_SAMPLES
    )


def extract_logmel(signal):
    mel_power = librosa.feature.melspectrogram(
        y=signal,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel_power,
        ref=np.max
    )

    return mel_db.astype(np.float32)

In [28]:
cycle_probabilities = []

model_patient.eval()

with torch.no_grad():

    for _, row in test_manifest_patient.iterrows():

        signal = load_processed_audio(
            row["audio_path"]
        )

        logmel = extract_logmel(signal)

        logmel = (
            logmel - logmel_mean
        ) / logmel_std

        x = torch.tensor(
            logmel,
            dtype=torch.float32
        ).unsqueeze(0).unsqueeze(0)

        x = x.to(device)

        logits = model_patient(x)

        probabilities = torch.softmax(
            logits,
            dim=1
        ).cpu().numpy()[0]

        cycle_probabilities.append(
            probabilities
        )

cycle_probabilities = np.asarray(
    cycle_probabilities,
    dtype=np.float32
)

print(
    "Probability matrix shape:",
    cycle_probabilities.shape
)

print(
    "Expected:",
    (len(test_manifest_patient), num_classes)
)

print(
    "First probability sum:",
    cycle_probabilities[0].sum()
)

Probability matrix shape: (1561, 8)
Expected: (1561, 8)
First probability sum: 1.0


In [29]:
probability_columns = [
    f"prob_{class_name}"
    for class_name in class_names
]

cycle_probability_df = pd.DataFrame(
    cycle_probabilities,
    columns=probability_columns
)

cycle_probability_df.insert(
    0,
    "patient_id",
    test_manifest_patient[
        "patient_id"
    ].to_numpy()
)

cycle_probability_df["true_label"] = (
    test_manifest_patient[
        "disease_label"
    ].to_numpy()
)

print(
    cycle_probability_df.head()
)

   patient_id  prob_Asthma  prob_Bronchiectasis  prob_Bronchiolitis  \
0         101     0.008635             0.003826            0.251323   
1         101     0.000215             0.000169            0.538953   
2         101     0.000571             0.000888            0.493316   
3         101     0.000176             0.000130            0.655411   
4         101     0.000713             0.000081            0.523310   

   prob_COPD  prob_Healthy  prob_LRTI  prob_Pneumonia  prob_URTI  true_label  
0   0.035413      0.183453   0.302630        0.084260   0.130459           7  
1   0.007750      0.128234   0.129588        0.005211   0.189880           7  
2   0.025993      0.089278   0.150578        0.015380   0.223995           7  
3   0.009163      0.091274   0.117227        0.005472   0.121147           7  
4   0.009125      0.129200   0.189199        0.008865   0.139506           7  


In [30]:
patient_probability_df = (
    cycle_probability_df
    .groupby("patient_id")[
        probability_columns
    ]
    .mean()
    .reset_index()
)

patient_probability_df["predicted_label"] = (
    patient_probability_df[
        probability_columns
    ]
    .to_numpy()
    .argmax(axis=1)
)

patient_probability_df["predicted_disease"] = (
    patient_probability_df[
        "predicted_label"
    ].map(
        dict(enumerate(class_names))
    )
)

print(
    patient_probability_df.head(10)
)

   patient_id  prob_Asthma  prob_Bronchiectasis  prob_Bronchiolitis  \
0         101     0.002820             0.005589            0.288779   
1         106     0.001882             0.087999            0.011705   
2         107     0.000680             0.074557            0.129379   
3         119     0.001410             0.003332            0.029994   
4         121     0.000230             0.011547            0.039150   
5         122     0.003678             0.005100            0.006809   
6         125     0.000435             0.024496            0.033111   
7         145     0.000237             0.043111            0.015378   
8         147     0.000264             0.189274            0.047749   
9         150     0.001695             0.007635            0.445614   

   prob_COPD  prob_Healthy  prob_LRTI  prob_Pneumonia  prob_URTI  \
0   0.129489      0.206614   0.088934        0.076854   0.200922   
1   0.566311      0.010963   0.001794        0.312355   0.006989   
2   0.517302  

In [31]:
patient_probability_df = patient_probability_df.merge(
    patient_truth_df[
        [
            "patient_id",
            "disease_label",
            "diagnosis"
        ]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

patient_probability_df = (
    patient_probability_df
    .rename(
        columns={
            "disease_label": "true_label",
            "diagnosis": "true_disease"
        }
    )
)

print(
    patient_probability_df[
        [
            "patient_id",
            "true_label",
            "true_disease",
            "predicted_label",
            "predicted_disease"
        ]
    ].head(10)
)

   patient_id  true_label true_disease  predicted_label predicted_disease
0         101           7         URTI                2     Bronchiolitis
1         106           3         COPD                3              COPD
2         107           3         COPD                3              COPD
3         119           7         URTI                4           Healthy
4         121           4      Healthy                7              URTI
5         122           6    Pneumonia                3              COPD
6         125           4      Healthy                7              URTI
7         145           3         COPD                3              COPD
8         147           3         COPD                3              COPD
9         150           7         URTI                2     Bronchiolitis


In [32]:
patient_probability_df = patient_probability_df.merge(
    patient_truth_df[
        [
            "patient_id",
            "disease_label",
            "diagnosis"
        ]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

patient_probability_df = (
    patient_probability_df
    .rename(
        columns={
            "disease_label": "true_label",
            "diagnosis": "true_disease"
        }
    )
)

print(
    patient_probability_df[
        [
            "patient_id",
            "true_label",
            "true_disease",
            "predicted_label",
            "predicted_disease"
        ]
    ].head(10)
)

   patient_id  true_label  true_label true_disease true_disease  \
0         101           7           7         URTI         URTI   
1         106           3           3         COPD         COPD   
2         107           3           3         COPD         COPD   
3         119           7           7         URTI         URTI   
4         121           4           4      Healthy      Healthy   
5         122           6           6    Pneumonia    Pneumonia   
6         125           4           4      Healthy      Healthy   
7         145           3           3         COPD         COPD   
8         147           3           3         COPD         COPD   
9         150           7           7         URTI         URTI   

   predicted_label predicted_disease  
0                2     Bronchiolitis  
1                3              COPD  
2                3              COPD  
3                4           Healthy  
4                7              URTI  
5                3          

In [34]:
# Start again from the probability aggregation table
patient_probability_df = (
    cycle_probability_df
    .groupby("patient_id")[probability_columns]
    .mean()
    .reset_index()
)

# Patient-level predicted class
patient_probability_df["predicted_label"] = (
    patient_probability_df[probability_columns]
    .to_numpy()
    .argmax(axis=1)
)

patient_probability_df["predicted_disease"] = (
    patient_probability_df["predicted_label"]
    .map(dict(enumerate(class_names)))
)

# Add ground truth using ONLY the required columns
patient_truth_clean = (
    patient_truth_df[
        ["patient_id", "disease_label", "diagnosis"]
    ]
    .rename(
        columns={
            "disease_label": "true_label",
            "diagnosis": "true_disease"
        }
    )
    .drop_duplicates("patient_id")
)

patient_probability_df = patient_probability_df.merge(
    patient_truth_clean,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", patient_probability_df.shape)

print(
    patient_probability_df[
        [
            "patient_id",
            "true_label",
            "true_disease",
            "predicted_label",
            "predicted_disease"
        ]
    ].head(10)
)

Shape: (25, 13)
   patient_id  true_label true_disease  predicted_label predicted_disease
0         101           7         URTI                2     Bronchiolitis
1         106           3         COPD                3              COPD
2         107           3         COPD                3              COPD
3         119           7         URTI                4           Healthy
4         121           4      Healthy                7              URTI
5         122           6    Pneumonia                3              COPD
6         125           4      Healthy                7              URTI
7         145           3         COPD                3              COPD
8         147           3         COPD                3              COPD
9         150           7         URTI                2     Bronchiolitis


In [35]:
y_true_probability = (
    patient_probability_df["true_label"]
    .to_numpy(dtype=np.int64)
)

y_pred_probability = (
    patient_probability_df["predicted_label"]
    .to_numpy(dtype=np.int64)
)

print("y_true shape:", y_true_probability.shape)
print("y_pred shape:", y_pred_probability.shape)

print("y_true dtype:", y_true_probability.dtype)
print("y_pred dtype:", y_pred_probability.dtype)

print("Number of patients:", len(y_true_probability))

assert y_true_probability.ndim == 1
assert y_pred_probability.ndim == 1
assert len(y_true_probability) == len(y_pred_probability)

prob_accuracy = accuracy_score(
    y_true_probability,
    y_pred_probability
)

prob_balanced_accuracy = balanced_accuracy_score(
    y_true_probability,
    y_pred_probability
)

prob_macro_precision = precision_score(
    y_true_probability,
    y_pred_probability,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

prob_macro_recall = recall_score(
    y_true_probability,
    y_pred_probability,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

prob_macro_f1 = f1_score(
    y_true_probability,
    y_pred_probability,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

prob_weighted_f1 = f1_score(
    y_true_probability,
    y_pred_probability,
    average="weighted",
    zero_division=0
)

print()
print(f"Accuracy:          {prob_accuracy:.6f}")
print(f"Balanced Accuracy: {prob_balanced_accuracy:.6f}")
print(f"Macro Precision:   {prob_macro_precision:.6f}")
print(f"Macro Recall:      {prob_macro_recall:.6f}")
print(f"Macro F1:          {prob_macro_f1:.6f}")
print(f"Weighted F1:       {prob_weighted_f1:.6f}")

y_true shape: (25,)
y_pred shape: (25,)
y_true dtype: int64
y_pred dtype: int64
Number of patients: 25

Accuracy:          0.560000
Balanced Accuracy: 0.333333
Macro Precision:   0.132812
Macro Recall:      0.250000
Macro F1:          0.162069
Weighted F1:       0.482207


In [36]:
print(
    classification_report(
        y_true_probability,
        y_pred_probability,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0
    )
)

                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00         2
 Bronchiolitis       0.25      1.00      0.40         1
          COPD       0.81      1.00      0.90        13
       Healthy       0.00      0.00      0.00         5
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.00      0.00      0.00         1
          URTI       0.00      0.00      0.00         3

      accuracy                           0.56        25
     macro avg       0.13      0.25      0.16        25
  weighted avg       0.43      0.56      0.48        25



In [37]:
probability_cm = confusion_matrix(
    y_true_probability,
    y_pred_probability,
    labels=np.arange(num_classes)
)

probability_cm_df = pd.DataFrame(
    probability_cm,
    index=class_names,
    columns=class_names
)

print(probability_cm_df)

                Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  \
Asthma               0               0              0     0        0     0   
Bronchiectasis       0               0              0     2        0     0   
Bronchiolitis        0               0              1     0        0     0   
COPD                 0               0              0    13        0     0   
Healthy              0               0              1     0        0     0   
LRTI                 0               0              0     0        0     0   
Pneumonia            0               0              0     1        0     0   
URTI                 0               0              2     0        1     0   

                Pneumonia  URTI  
Asthma                  0     0  
Bronchiectasis          0     0  
Bronchiolitis           0     0  
COPD                    0     0  
Healthy                 0     4  
LRTI                    0     0  
Pneumonia               0     0  
URTI                    0

In [38]:
patient_aggregation_comparison = pd.DataFrame([
    {
        "aggregation_method": "Majority Vote",
        "accuracy": patient_majority_accuracy,
        "balanced_accuracy": patient_majority_balanced_accuracy,
        "macro_precision": patient_majority_macro_precision,
        "macro_recall": patient_majority_macro_recall,
        "macro_f1": patient_majority_macro_f1,
        "weighted_f1": patient_majority_weighted_f1
    },
    {
        "aggregation_method": "Mean Probability",
        "accuracy": prob_accuracy,
        "balanced_accuracy": prob_balanced_accuracy,
        "macro_precision": prob_macro_precision,
        "macro_recall": prob_macro_recall,
        "macro_f1": prob_macro_f1,
        "weighted_f1": prob_weighted_f1
    }
])

print(
    patient_aggregation_comparison.to_string(
        index=False
    )
)

aggregation_method  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
     Majority Vote      0.56           0.333333         0.132812          0.25  0.162069     0.482207
  Mean Probability      0.56           0.333333         0.132812          0.25  0.162069     0.482207


In [39]:
patient_aggregation_ranking = (
    patient_aggregation_comparison
    .sort_values(
        by=[
            "macro_f1",
            "balanced_accuracy"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

patient_aggregation_ranking.insert(
    0,
    "rank",
    np.arange(
        1,
        len(patient_aggregation_ranking) + 1
    )
)

print(
    patient_aggregation_ranking.to_string(
        index=False
    )
)

 rank aggregation_method  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
    1      Majority Vote      0.56           0.333333         0.132812          0.25  0.162069     0.482207
    2   Mean Probability      0.56           0.333333         0.132812          0.25  0.162069     0.482207


In [40]:
aggregation_comparison_path = (
    PATIENT_LEVEL_DIR /
    "patient_aggregation_comparison.csv"
)

patient_aggregation_ranking.to_csv(
    aggregation_comparison_path,
    index=False
)

print(
    "Saved:",
    aggregation_comparison_path
)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/patient_aggregation_comparison.csv


In [41]:
majority_predictions_path = (
    PATIENT_LEVEL_DIR /
    "patient_majority_predictions.csv"
)

patient_majority_df.to_csv(
    majority_predictions_path,
    index=False
)

print(
    "Saved:",
    majority_predictions_path
)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/patient_majority_predictions.csv


In [42]:
probability_predictions_path = (
    PATIENT_LEVEL_DIR /
    "patient_probability_predictions.csv"
)

patient_probability_df.to_csv(
    probability_predictions_path,
    index=False
)

print(
    "Saved:",
    probability_predictions_path
)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/patient_probability_predictions.csv


In [43]:
majority_cm_path = (
    PATIENT_LEVEL_DIR /
    "majority_vote_confusion_matrix.csv"
)

probability_cm_path = (
    PATIENT_LEVEL_DIR /
    "mean_probability_confusion_matrix.csv"
)

patient_majority_cm_df.to_csv(
    majority_cm_path
)

probability_cm_df.to_csv(
    probability_cm_path
)

print("Saved:", majority_cm_path)
print("Saved:", probability_cm_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/majority_vote_confusion_matrix.csv
Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/mean_probability_confusion_matrix.csv


In [44]:
best_patient_method = (
    patient_aggregation_ranking.iloc[0]
)

patient_summary = {
    "experiment": "Patient-level prediction",
    "base_model": "Lightweight Mel CNN",
    "test_patients": int(
        test_manifest_patient["patient_id"].nunique()
    ),
    "test_cycles": int(
        len(test_manifest_patient)
    ),
    "aggregation_methods": [
        "Majority Vote",
        "Mean Probability"
    ],
    "best_aggregation_method": (
        best_patient_method[
            "aggregation_method"
        ]
    ),
    "best_macro_f1": float(
        best_patient_method["macro_f1"]
    ),
    "best_balanced_accuracy": float(
        best_patient_method[
            "balanced_accuracy"
        ]
    ),
    "patient_disjoint_evaluation": True
}

summary_path = (
    PATIENT_LEVEL_DIR /
    "patient_level_experiment_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        patient_summary,
        f,
        indent=2
    )

print(
    "Saved:",
    summary_path
)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/patient_level_prediction/patient_level_experiment_summary.json


## Patient-Level Prediction — Interpretation

The patient-level experiment addresses an important characteristic of
the respiratory disease classification problem: disease diagnosis is
associated with the patient, whereas the CNN operates on individual
respiratory cycles.

Therefore, cycle-level predictions were aggregated to obtain a single
disease prediction for each patient.

Two aggregation strategies were evaluated:

1. Majority voting, where the most frequently predicted disease across
   the patient's respiratory cycles is selected.
2. Mean probability aggregation, where the class probabilities across
   all respiratory cycles are averaged before selecting the highest
   probability class.

The two approaches are compared using accuracy, balanced accuracy,
macro precision, macro recall, macro F1 and weighted F1.

Balanced accuracy and macro F1 are treated as the primary evaluation
criteria because the patient-level test set is highly imbalanced.

The patient-level experiment contains only 25 test patients. Several
disease classes have very small or zero representation in this split.
Therefore, patient-level metrics, especially for minority diseases,
should be interpreted cautiously.

The patient-level results provide a more clinically appropriate
evaluation than treating individual respiratory cycles as independent
patients.